## This is an experiment of MIRACL training for the following scenario:
- Simple SC problems
- Scenario: various weights, demands, and parameters; shared supply chain network

In [ ]:
from gymnasium.wrappers import TimeLimit
import gymnasium as gym

from algorithms.miracl import MIRACL, MIRACLConfig #Use algo from local repo
import ray #v2.3.1 
from ray import air, tune
from ray.tune.registry import register_env
from ray.air import Checkpoint

from messiah import State, Runner, History, settings
from messiah.state.components import Node, Edge
from messiah.series.basic import ConstantSeries, RandomSeries, DataSeries
from messiah.agents.naive import FixedQuantityProcessing
from messiah.agents.basic import MaxAvailableProcessing, MaxRequiredProcessing
from messiah.config import settings

import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
import seaborn as sns
import pygmo as pg

import tempfile


from utils.plot_utils import PlotUtils

In [ ]:
#Import the autogenerated tasks for metatraining
from environments import simple_environment

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
miracl_simple_sc = simple_environment.SimpleSC()

In [ ]:
ray.shutdown()
ray.init()

register_env(
    'miracl_sim_train',
    lambda env_cfg: miracl_simple_sc
)
    
rollout_fragment_length = 32

## Adaptation and MetaLearning Phase

In [ ]:
config = (
    MIRACLConfig()
    .rollouts(
        num_rollout_workers=4, rollout_fragment_length=rollout_fragment_length
    )
    .framework("torch")
    .environment("miracl_sim_train", clip_actions=True)
    .training(
        inner_adaptation_steps=4,
        maml_optimizer_steps=10,
        gamma=0.99,
        lambda_=1.0,
        lr=0.001,
        vf_loss_coeff=0.5,
        inner_lr=0.03,
        use_meta_env=True,
        clip_param=0.3,
        kl_target=0.01,
        kl_coeff=0.001,
        model=dict(fcnet_hiddens=[64, 64]),
        train_batch_size=rollout_fragment_length,
    )
    .evaluation(
        evaluation_num_workers=1,
        evaluation_interval=100,
        enable_async_evaluation=True,
    )
)

In [ ]:
num_iterations = 250 #for exist it is 500 iterations for 1e6 timesteps
results = {}
runs = 10

for i in range(runs):
    print(f"RUNNING-{i}")
    tuner = tune.Tuner(
        MIRACL,
        param_space=config.to_dict(),
        run_config=air.RunConfig(
            stop={"training_iteration": num_iterations},
            failure_config=air.FailureConfig(fail_fast="raise"),
            log_to_file=True
        )
    )
    results[i] = tuner.fit()

    #save the best checkpoint for each run
    best_result = results[i].get_best_result()
    best_result_path = f"training/miracl_sim_data/{i}miracl_sim_train_metapolicy"

    # Ensure the directory exists
    os.makedirs(os.path.dirname(best_result_path), exist_ok=True)
    best_result.checkpoint.to_directory(best_result_path)
    print(f"Best checkpoint for run {i} stored in: {best_result_path}")

In [ ]:
#Print the plot
plotter = PlotUtils(results)
plotter.plot_reward()

Now, the training data including meta-parameters are stored and can be used for fine-tuning phase.

## Fine-tuning using Stable-baseline3

Use the parameters from the Appendix file.

In [ ]:
from fine_tuning.simple_test_env import TestSimpleSC

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.ppo.policies import MlpPolicy
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

In [ ]:
#Register the environment with Gym
TestSimpleSC()

gym.envs.register(
    id='testSC',
    entry_point=TestSimpleSC
)

In [ ]:
#Check if environment is successfully registered
env_name = 'testSC'

try:
    # Attempt to create an instance of the environment
    env = gym.make(env_name)
    print(f"Environment '{env_name}' is successfully registered in Gymnasium.")
except gym.error.UnregisteredEnv:
    print(f"Environment '{env_name}' is not registered in Gymnasium.")
except Exception as e:
    print(f"An error occurred: {e}")

Automatically run multiple env, where an env is associated to a weight vector to produce PF set

In [ ]:
#From stable_baselines3.common.evaluation import evaluate_policy
from fine_tuning.evaluate_normalise import eval_nlocal

# define evaluate function
def evaluate(model, env, eval_episodes=50, verbose=False):
    mean_reward, std_reward, mean_reward_profit, mean_reward_emission, mean_reward_equity = eval_nlocal(model, env, n_eval_episodes=eval_episodes)
    if verbose:
        print(f"mean_reward:{mean_reward:.2f} +/- {std_reward:.2f}")
    return mean_reward, std_reward, mean_reward_profit, mean_reward_emission, mean_reward_equity

In [ ]:
#Training parameters for each env
total_steps = 4500
record_every = 250
eval_episodes = 50
dm_steps = 10

In [ ]:
#Function to apply dm during fine-tuning
def adjust_weight_dm(archived_rewards, weight_list, delta=1.05, steps=1):
    archived_rewards = np.array(archived_rewards)
    weight_list = np.array(weight_list)

    for step in range(steps):
        for idx in range(len(weight_list)):
            #Pick the most recent reward for this solution
            #Assumes archive is ordered, 1 reward per solution per step
            vector_reward = archived_rewards[-len(weight_list) + idx]

            # Find closest neighbor (excluding self)
            distances = np.linalg.norm(archived_rewards - vector_reward, axis=1)
            distances[-len(weight_list) + idx] = np.inf  #Exclude self from comparison
            closest_idx = np.argmin(distances)
            closest_non_dominated = archived_rewards[closest_idx]

            weights = weight_list[idx].copy()
            for j in range(len(weights)):
                if vector_reward[j] >= closest_non_dominated[j]:  #Maximisation
                    weights[j] *= delta
                else:
                    weights[j] /= delta

            # Normalize
            weight_list[idx] = weights / max(np.sum(weights), 1e-8)

    return weight_list

In [ ]:
'''All train data is automatically exported into csv.
Change file names for not being OVERWRITTEN.
Do fine-tuning one-by-one for each meta-policy due to CPU limitation'''

runs = 10 #Meta-policy running number
for k in range(runs):
#Restart ray
    ray.shutdown()
    ray.init()
    register_env(
        'miracl_sim_train',
        lambda env_cfg: miracl_simple_sc
    )

    rollout_fragment_length = 32

    #restore checkpoint
    checkpoint = Checkpoint.from_directory(f"training/miracl_sim_data/{i}miracl_sim_train_metapolicy")
    trainer = (
         MIRACLConfig()
        .rollouts(
            num_rollout_workers=4, rollout_fragment_length=rollout_fragment_length
        )
        .framework("torch")
        .environment("miracl_sim_train", clip_actions=True)
        .training(
            inner_adaptation_steps=4,
            maml_optimizer_steps=10,
            gamma=0.99,
            lambda_=1.0,
            lr=0.001,
            vf_loss_coeff=0.5,
            inner_lr=0.03,
            use_meta_env=True,
            clip_param=0.3,
            kl_target=0.01,
            kl_coeff=0.001,
            model=dict(fcnet_hiddens=[64, 64]),
            train_batch_size=rollout_fragment_length,
        )
        .evaluation(
            evaluation_num_workers=1,
            evaluation_interval=100,
            enable_async_evaluation=True,
        )
        .build()
    )
    trainer.restore(checkpoint.to_directory())
    restored_trainer = trainer

    #fine tuning phase
    #weight_num = 20 #desired number of solutions
    episode_num = np.linspace(record_every, total_steps, int(total_steps/record_every))
    # Train the agent

    print(f'RUNNING NUMBER-{k}')
    start = time.time()

    weight_list = np.array([[0,0,1],[0,0.2,0.8],[0,0.4,0.6],[0,0.6,0.4],[0,0.8,0.2],
                         [0,1,0],[0.2,0,0.8],[0.2,0.2,0.6],[0.2,0.4,0.4],[0.2,0.6,0.2],
                         [0.2,0.8,0],[0.4,0,0.6],[0.4,0.2,0.4],[0.4,0.4,0.2],[0.4,0.6,0],
                         [0.6,0,0.4],[0.6,0.2,0.2],[0.6,0.4,0],[0.8,0,0.2],[0.8,0.2,0],
                         [1,0,0]])
    
    total_intervals = int(total_steps/record_every) + dm_steps
    mean_reward_history = np.zeros((weight_list.shape[0], total_intervals))
    std_reward_history = np.zeros_like(mean_reward_history)
    mean_reward_profit_hist = np.zeros_like(mean_reward_history)
    mean_reward_emission_hist = np.zeros_like(mean_reward_history)
    mean_reward_equity_hist = np.zeros_like(mean_reward_history)

    # Ensure model saving directory exists
    model_save_path = f"fine_tuning/miracl_sim_data/"
    os.makedirs(model_save_path, exist_ok=True)

    for n, weights in enumerate(weight_list):
        #weights = weight_list[n]
        env.weights = weights #update weights
        print(f'start training for solution-{n}, weights:',weights)

        #refrech the model so that each weight runs independently
        ray_policy = restored_trainer.get_policy().model.state_dict()
        model = PPO(MlpPolicy, env="testSC", verbose=0)

        # Filter the keys in ray_policy to match the Stable-Baselines3 policy
        filtered_ray_policy = {
            key: value for key, value in ray_policy.items() if key in model.policy.state_dict()
        }

        # Update and load the state dictionary
        policy_state_dict = model.policy.state_dict()
        policy_state_dict.update(filtered_ray_policy)
        model.policy.load_state_dict(policy_state_dict)

        for i in range(int(total_steps/record_every)):
            model.learn(total_timesteps=record_every)
            mean_reward, std_reward, mean_reward_profit, mean_reward_emission, mean_reward_equity = evaluate(model, env, eval_episodes=eval_episodes, verbose=True)
            mean_reward_history[n,i] = mean_reward
            std_reward_history[n,i] = std_reward
            mean_reward_profit_hist[n,i] = mean_reward_profit
            mean_reward_emission_hist[n,i] = mean_reward_emission
            mean_reward_equity_hist[n,i] = mean_reward_equity
            #print('weights:',weights) #for checking weights are similar across i
            print('vector_rewards: ',mean_reward_profit, mean_reward_emission, mean_reward_equity)
        print()

        # Save the trained model for this solution `n`
        model_file = f"{model_save_path}{k}_solution_{n}.zip"
        model.save(model_file)
        
    # === DM Phase AFTER Main Training ===
    print(f"\n>>> STARTING DM PHASE FOR RUN-{k} <<<\n")
    archived_rewards = []

    # Rebuild archive from last reward snapshot
    for n in range(weight_list.shape[0]):
        final_reward = np.array([
            mean_reward_profit_hist[n, -1],
            mean_reward_emission_hist[n, -1],
            mean_reward_equity_hist[n, -1],
        ])
        archived_rewards.append(final_reward)

    for n in range(len(weight_list)):
        env.weights = weight_list[n]
        print(f'DM-Finetuning solution-{n}, initial weights: {weight_list[n]}')

        # Load trained model
        model_file = f"{model_save_path}{k}_solution_{n}.zip"
        model = PPO.load(model_file, env=env, verbose=0)
        
        dm_start_index = int(total_steps / record_every)
        for step in range(dm_steps):
            model.learn(total_timesteps=50)

            mean_reward, std_reward, mean_reward_profit, mean_reward_emission, mean_reward_equity = evaluate(
                model, env, eval_episodes=eval_episodes, verbose=False)
            vector_reward = np.array([mean_reward_profit, mean_reward_emission, mean_reward_equity])
            idx = dm_start_index + step
            mean_reward_history[n, idx] = mean_reward
            std_reward_history[n, idx] = std_reward
            mean_reward_profit_hist[n, idx] = mean_reward_profit
            mean_reward_emission_hist[n, idx] = mean_reward_emission
            mean_reward_equity_hist[n, idx] = mean_reward_equity
            archived_rewards.append(vector_reward)

            # Adjust weights using dm
            weight_list = adjust_weight_dm(archived_rewards, weight_list, delta=1.05, steps=1)
            env.weights = weight_list[n]
            print(f"DM-Step {step+1}, updated weight for solution-{n}: {weight_list[n]}")
        print('vector_rewards_dm: ',mean_reward_profit, mean_reward_emission, mean_reward_equity)

        # Save DM-refined model
        dm_model_file = f"{model_save_path}{k}_solution_{n}_dm.zip"
        model.save(dm_model_file)
        print(f"Saved DM-finetuned model at {dm_model_file}")


    print(f"Saved trained model for running-{k} solution-{n} at {model_file}")
    print('Training time:', time.time() - start, 'seconds')
    print('='*40)
    print()

    #export result into csv
    df_weight_list = pd.DataFrame(weight_list)
    df_mean_reward = pd.DataFrame(mean_reward_history) #rows= n, columns = i
    df_std_reward = pd.DataFrame(std_reward_history)
    df_profit_reward = pd.DataFrame(mean_reward_profit_hist)
    df_emission_reward = pd.DataFrame(mean_reward_emission_hist)
    df_equity_reward = pd.DataFrame(mean_reward_equity_hist)

    df_weight_list.to_csv(f'fine_tuning/miracl_sim_data/{k}.miracl_sim_weight_list.csv',index=False)
    df_mean_reward.to_csv(f'fine_tuning/miracl_sim_data/{k}.miracl_sim_mean_reward.csv',index=False)
    df_std_reward.to_csv(f'fine_tuning/miracl_sim_data/{k}.miracl_sim_std_reward.csv',index=False)
    df_profit_reward.to_csv(f'fine_tuning/miracl_sim_data/{k}.miracl_sim_profit_reward.csv',index=False)
    df_emission_reward.to_csv(f'fine_tuning/miracl_sim_data/{k}.miracl_sim_emission_reward.csv',index=False)
    df_equity_reward.to_csv(f'fine_tuning/miracl_sim_data/{k}.miracl_sim_equity_reward.csv',index=False)
    
print('ALL RUNNINGS DONE!')

#### Reload training data and fine-tuning results from csv

## Performance Measurement

In [ ]:
runs = 10 #number of runs to load all csv data

#initiate lists
weight_list = []
mean_reward = []
std_reward = []
profit_reward = []
emission_reward = []
equity_reward = []

for k in range(runs):
    df_weight_list = pd.read_csv(f'fine_tuning/miracl_sim_data/{k}.miracl_sim_weight_list.csv')
    df_mean_reward = pd.read_csv(f'fine_tuning/miracl_sim_data/{k}.miracl_sim_mean_reward.csv')
    df_std_reward = pd.read_csv(f'fine_tuning/miracl_sim_data/{k}.miracl_sim_std_reward.csv')
    df_profit_reward = pd.read_csv(f'fine_tuning/miracl_sim_data/{k}.miracl_sim_profit_reward.csv')
    df_emission_reward = pd.read_csv(f'fine_tuning/miracl_sim_data/{k}.miracl_sim_emission_reward.csv')
    df_equity_reward = pd.read_csv(f'fine_tuning/miracl_sim_data/{k}.miracl_sim_equity_reward.csv')

    weight_list.append(np.array(df_weight_list))
    mean_reward.append(np.array(df_mean_reward))
    std_reward.append(np.array(df_std_reward))
    profit_reward.append(np.array(df_profit_reward))
    emission_reward.append(np.array(df_emission_reward))
    equity_reward.append(np.array(df_equity_reward))
    
weight_list = np.array(weight_list)
mean_reward = np.array(mean_reward) #nest sequence: k -> n -> i
std_reward = np.array(std_reward)
profit_reward = np.array(profit_reward)
emission_reward = np.array(emission_reward)
equity_reward = np.array(equity_reward)

#### Hypervolume

In [ ]:
mean_reward.shape

In [ ]:
#set reference value
min_point = np.array([0,-2e5,-1e2]) #min point
max_point = np.array([4e5,0,0])
ref_point = np.array([4e5,0,0])

In [ ]:
#inverse values (this is because pygmo uses max ref point instead of min like in MORLD)
reversed_profit_reward, reversed_emission_reward, reversed_equity_reward = [np.empty(mean_reward.shape) for _ in range(3)]

for k in range(mean_reward.shape[0]): #runs
    for n in range(mean_reward.shape[1]): #weights
        for i in range(mean_reward.shape[2]): #iterations
            reversed_profit_reward[k,n,i] = max_point[0] - (profit_reward[k,n,i] - min_point[0])
            reversed_emission_reward[k,n,i] = max_point[1] - (emission_reward[k,n,i] - min_point[1])
            reversed_equity_reward[k,n,i] = max_point[2] - (equity_reward[k,n,i] - min_point[2])

In [ ]:
combined_rewards = np.stack([profit_reward,emission_reward,equity_reward],axis=-1)
reversed_combined_rewards = np.stack([reversed_profit_reward,reversed_emission_reward,reversed_equity_reward],axis=-1)

In [ ]:
combined_rewards.shape

In [ ]:
#calculate Hypervolume
# Initialize the hypervolume lists
final_h = []

# Calculate hypervolumes using pygmo's hypervolume function
for k in range(reversed_combined_rewards.shape[0]): #runs
    h = []
    for i in range(reversed_combined_rewards.shape[2]): #iterations
        t = []
        for n in range(reversed_combined_rewards.shape[1]): #weights
            clipped_point = np.minimum(reversed_combined_rewards[k,n,i], ref_point - 1e-5) #clip the points within reference point
            # Aggregate data for all weights j at the current stage
            t.append(clipped_point)
        hv = pg.hypervolume(t)
        ref_point = ref_point
        h.append(hv.compute(ref_point))
    final_h.append(h)

In [ ]:
#save hypervolume calculation into dataframe
df_h = pd.DataFrame(final_h)
df_h = df_h.T

df_h["mean"] = df_h.mean(axis=1)
df_h["std"] = df_h.std(axis=1)

In [ ]:
# Construct non-uniform x-axis (timesteps)
x = [i * 250 for i in range(18)] + [4500 + (i * 50) for i in range(10)]  # 18 + 10 = 28 points

# Calculate confidence interval bounds
df_h["upper"] = df_h["mean"] + 1.96 * (df_h["std"] / (10**0.5))
df_h["lower"] = df_h["mean"] - 1.96 * (df_h["std"] / (10**0.5))

# Plot
plt.plot(x, df_h["mean"], color="blue")
plt.fill_between(x, df_h["upper"], df_h["lower"], color="blue", alpha=0.1)
plt.xlabel("timesteps", fontsize=20)
plt.ylabel("hypervolume", fontsize=20)
plt.title("Hypervolume Progression with DM Phase", fontsize=14)
plt.tight_layout()
plt.show()

#### Sparsity

In [ ]:
#calculate sparsity for convengence plot
def calculate_sparsity_3d(pareto_front_3d):
    # Flatten the 3D array to 2D for sorting
    pareto_front = pareto_front_3d.reshape(-1, pareto_front_3d.shape[-1])

    # Sort the Pareto front based on the first objective
    sorted_pareto_front = pareto_front[pareto_front[:, 0].argsort()]

    # Calculate the squared distances between consecutive points
    squared_distances = np.sum(np.diff(sorted_pareto_front, axis=0) ** 2, axis=1)

    # Calculate the average squared distance (sparsity)
    sparsity = np.mean(squared_distances)

    return sparsity

In [ ]:
# Initialize the hypervolume lists
final_s = []

# Calculate hypervolumes using pygmo's hypervolume function
for k in range(combined_rewards.shape[0]): #runs
    s = []
    for i in range(combined_rewards.shape[2]): #iterations
        t = []
        for n in range(combined_rewards.shape[1]): #weights
        # Aggregate data for all weights j at the current stage
            t.append(combined_rewards[k,n,i])
        t = np.array(t)
        s.append(calculate_sparsity_3d(t))
    #print(len(h))
    final_s.append(s)

In [ ]:
#print sparsity
df_s = pd.DataFrame(final_s)
df_s = df_s.T

df_s["mean"] = df_s.mean(axis=1)
df_s["std"] = df_s.std(axis=1)

In [ ]:
#plot hypervolume
x = [i * 250 for i in range(18)] + [4500 + (i * 50) for i in range(10)]  # 18 + 10 = 28 points

df_s["upper"] = df_s["mean"] + 1.96*((df_s["std"])/(10**0.5))
df_s["lower"] = df_s["mean"] - 1.96*((df_s["std"])/(10**0.5))

plt.plot(x, df_s["mean"], color="blue")
plt.xlabel("timesteps", fontsize=20)
plt.ylabel("sparsity",fontsize=20)
plt.fill_between(x, df_s["upper"], df_s["lower"], color="blue", alpha=0.1)
plt.show()

#### Expected Utility Metric

In [ ]:
# Initialize the hypervolume lists
final_e = []

# Calculate hypervolumes using pygmo's hypervolume function
for k in range(combined_rewards.shape[0]): #runs
    e = []
    for i in range(combined_rewards.shape[2]): #iterations
        eum = 0
        for n in range(combined_rewards.shape[1]): #weights
        # Aggregate data for all weights j at the current stage
            reward_vector = combined_rewards[k,n,i]
            weights = weight_list[k,n]
            eum += np.dot(reward_vector,weights)
        
        e.append(eum/combined_rewards.shape[1]) #average over weights

    final_e.append(e)

In [ ]:
#print EUM
df_e = pd.DataFrame(final_e)
df_e = df_e.T
df_e["mean"] = df_e.mean(axis=1)
df_e["std"] = df_e.std(axis=1)

In [ ]:
#plot hypervolume
x = [i * 250 for i in range(18)] + [4500 + (i * 50) for i in range(10)]  # 18 + 10 = 28 points

df_e["upper"] = df_e["mean"] + 1.96*((df_e["std"])/(10**0.5))
df_e["lower"] = df_e["mean"] - 1.96*((df_e["std"])/(10**0.5))

plt.plot(x, df_e["mean"], color="blue")
plt.xlabel("timesteps", fontsize=20)
plt.ylabel("EUM",fontsize=20)
plt.fill_between(x, df_e["upper"], df_e["lower"], color="blue", alpha=0.1)
plt.show()